# Qwen3-8B QLoRA Fine-tuning on Pile of Law Dataset

This notebook contains the complete pipeline to fine-tune the Qwen3-8B model (or Qwen2.5-7B) on Hugging Face's **Pile of Law** dataset using **QLoRA (4-bit quantization with LoRA adapters)**.

### Hardware Requirement
- Recommended: **Kaggle GPU P100** or **dual T4 GPUs** (using PyTorch DDP/Accelerate or single-GPU training).

## 1. Install Dependencies

In [ ]:
!pip install -q 'transformers<4.49' peft bitsandbytes accelerate datasets sentencepiece protobuf


## 2. Imports and Initialization

In [ ]:
import os
import math
import random
import logging
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_scheduler
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from tqdm.notebook import tqdm

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("KaggleFineTune")

## 3. Configuration Setup

In [ ]:
config = {
    "model": {
        "name": "Qwen/Qwen3-8B",         # Change to "Qwen/Qwen2.5-7B" if desired
        "max_length": 1024,             # Max token context size
        "use_qlora": True
    },
    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "bias": "none",
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj"
        ]
    },
    "data": {
        "dataset_name": "pile-of-law/pile-of-law",
        "subset": "r_legaladvice",       # Subset choice (e.g. "r_legaladvice", "courtListener_opinions")
        "text_col": "text",
        "max_train_samples": 5000,      # Sliced for demonstration speed
        "max_val_samples": 500
    },
    "training": {
        "batch_size": 2,
        "gradient_accumulation_steps": 8,  # Effective batch size = 2 * 8 = 16
        "epochs": 1,
        "learning_rate": 2e-4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.03,
        "seed": 42,
        "device": "auto",
        "save_dir": "./outputs",
        "dry_run": False                 # Toggle True to verify execution in seconds
    }
}

## 4. Helper Functions (Seed and Device)

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    logger.info(f"Seed set to {seed}")

def get_device(device_setting):
    if device_setting == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        elif torch.backends.mps.is_available():
            return torch.device("mps")
        else:
            return torch.device("cpu")
    return torch.device(device_setting)

## 5. Dataset Definition

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, dataset_name, subset, split, tokenizer, max_length=1024, max_samples=None, text_col="text"):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.text_col = text_col
        
        logger.info(f"Loading dataset '{dataset_name}' (subset: '{subset}', split: '{split}')...")
        raw_dataset = load_dataset(dataset_name, name=subset, split=split, trust_remote_code=True)
        
        if max_samples and max_samples < len(raw_dataset):
            logger.info(f"Limiting '{split}' split to first {max_samples} samples.")
            self.samples = raw_dataset.select(range(max_samples))
        else:
            self.samples = raw_dataset
            
        logger.info(f"Loaded {len(self.samples)} samples for '{split}' split.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        text = sample[self.text_col] or ""
        
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        
        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]
        labels = input_ids.copy()
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

def causal_lm_collate_fn(batch, pad_token_id):
    input_ids = [torch.tensor(item["input_ids"]) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"]) for item in batch]
    labels = [torch.tensor(item["labels"]) for item in batch]
    
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids, batch_first=True, padding_value=pad_token_id
    )
    padded_attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_mask, batch_first=True, padding_value=0
    )
    padded_labels = torch.nn.utils.rnn.pad_sequence(
        labels, batch_first=True, padding_value=-100
    )
    
    return {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention_mask,
        "labels": padded_labels
    }

def create_dataloader(dataset_name, subset, split, tokenizer, batch_size, max_length, max_samples=None, shuffle=True, text_col="text"):
    dataset = CustomDataset(
        dataset_name=dataset_name,
        subset=subset,
        split=split,
        tokenizer=tokenizer,
        max_length=max_length,
        max_samples=max_samples,
        text_col=text_col
    )
    
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=lambda b: causal_lm_collate_fn(b, pad_token_id)
    )
    return dataloader

## 6. Model Loading Function

In [ ]:
def load_qlora_model_and_tokenizer(model_name, lora_config, use_qlora=True, device="auto"):
    logger.info(f"Loading model '{model_name}'...")
    
    bnb_config = None
    if use_qlora:
        if torch.cuda.is_available():
            logger.info("CUDA available. Building NF4 4-bit config...")
            compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=compute_dtype
            )
        else:
            logger.warning("CUDA not available. QLoRA quantization bypassed; loading default precision.")

    device_map = "auto" if device == "auto" else device
    if not torch.cuda.is_available() and device_map == "auto":
        device_map = "cpu"
        
    torch_dtype = torch.float32
    if torch.cuda.is_available():
        torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map=device_map,
        torch_dtype=torch_dtype,
        trust_remote_code=True
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    if use_qlora and bnb_config is not None:
        model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        r=lora_config.get("r", 16),
        lora_alpha=lora_config.get("alpha", 32),
        target_modules=lora_config.get("target_modules", ["q_proj", "k_proj", "v_proj", "o_proj"]),
        lora_dropout=lora_config.get("dropout", 0.05),
        bias=lora_config.get("bias", "none"),
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, peft_config)
    model.config.use_cache = False
    
    model.print_trainable_parameters()
    return model, tokenizer

## 7. Training and Evaluation Loops

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, gradient_accumulation_steps=1, dry_run=False):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    
    num_steps = len(dataloader)
    progress_bar = tqdm(enumerate(dataloader), total=num_steps, desc="Training")
    
    actual_steps = 0
    for step, batch in progress_bar:
        if dry_run and step >= 3:
            break
            
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / gradient_accumulation_steps
        
        loss.backward()
        
        total_loss += loss.item() * gradient_accumulation_steps
        actual_steps += 1
        
        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == num_steps:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            optimizer.zero_grad()
            
        progress_bar.set_postfix({"loss": f"{loss.item() * gradient_accumulation_steps:.4f}"})
        
    return total_loss / max(actual_steps, 1)

def evaluate_model(model, dataloader, device, dry_run=False):
    model.eval()
    total_loss = 0.0
    
    num_steps = len(dataloader)
    progress_bar = tqdm(enumerate(dataloader), total=num_steps, desc="Evaluating")
    
    actual_steps = 0
    with torch.no_grad():
        for step, batch in progress_bar:
            if dry_run and step >= 3:
                break
                
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            total_loss += loss.item()
            actual_steps += 1
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
            
    avg_loss = total_loss / max(actual_steps, 1)
    try:
        perplexity = math.exp(avg_loss)
    except OverflowError:
        perplexity = float("inf")
        
    return {
        "loss": avg_loss,
        "perplexity": perplexity
    }

def train_model(model, train_dataloader, val_dataloader, optimizer, scheduler, device, epochs, gradient_accumulation_steps, save_dir, dry_run=False):
    best_val_loss = float("inf")
    
    for epoch in range(epochs):
        logger.info(f"=== Epoch {epoch + 1}/{epochs} ===")
        train_loss = train_epoch(model, train_dataloader, optimizer, scheduler, device, gradient_accumulation_steps, dry_run)
        logger.info(f"Epoch {epoch + 1} average training loss: {train_loss:.4f}")
        
        logger.info("Evaluating model...")
        val_metrics = evaluate_model(model, val_dataloader, device, dry_run)
        val_loss = val_metrics["loss"]
        ppl = val_metrics["perplexity"]
        logger.info(f"Validation loss: {val_loss:.4f} | Perplexity: {ppl:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, "best_model_checkpoint")
            logger.info(f"Saving new best adapters to {save_path}...")
            model.save_pretrained(save_path)
            
    return best_val_loss

## 8. Run Training Pipeline

In [ ]:
def run_finetuning(cfg):
    set_seed(cfg["training"]["seed"])
    device = get_device(cfg["training"]["device"])
    logger.info(f"Running on device: {device}")
    
    # Load model & tokenizer
    model, tokenizer = load_qlora_model_and_tokenizer(
        model_name=cfg["model"]["name"],
        lora_config=cfg["lora"],
        use_qlora=cfg["model"]["use_qlora"],
        device=cfg["training"]["device"]
    )
    
    is_dry_run = cfg["training"].get("dry_run", False)
    max_train = 10 if is_dry_run else cfg["data"]["max_train_samples"]
    max_val = 5 if is_dry_run else cfg["data"]["max_val_samples"]
    
    # Dataloaders
    train_dataloader = create_dataloader(
        dataset_name=cfg["data"]["dataset_name"],
        subset=cfg["data"]["subset"],
        split="train",
        tokenizer=tokenizer,
        batch_size=cfg["training"]["batch_size"],
        max_length=cfg["model"]["max_length"],
        max_samples=max_train,
        shuffle=True,
        text_col=cfg["data"]["text_col"]
    )
    
    val_dataloader = create_dataloader(
        dataset_name=cfg["data"]["dataset_name"],
        subset=cfg["data"]["subset"],
        split="validation",
        tokenizer=tokenizer,
        batch_size=cfg["training"]["batch_size"],
        max_length=cfg["model"]["max_length"],
        max_samples=max_val,
        shuffle=False,
        text_col=cfg["data"]["text_col"]
    )
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["training"]["learning_rate"]),
        weight_decay=float(cfg["training"]["weight_decay"])
    )
    
    # Scheduler calculations
    epochs = cfg["training"]["epochs"]
    grad_accum = cfg["training"]["gradient_accumulation_steps"]
    num_update_steps_per_epoch = len(train_dataloader) // grad_accum
    num_update_steps_per_epoch = max(num_update_steps_per_epoch, 1)
    max_steps = epochs * num_update_steps_per_epoch
    warmup = int(max_steps * cfg["training"]["warmup_ratio"])
    
    scheduler = get_scheduler(
        name="linear",
        optimizer=optimizer,
        num_warmup_steps=warmup,
        num_training_steps=max_steps
    )
    
    logger.info("Starting orchestration loop...")
    best_loss = train_model(
        model=model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=epochs,
        gradient_accumulation_steps=grad_accum,
        save_dir=cfg["training"]["save_dir"],
        dry_run=is_dry_run
    )
    logger.info(f"Orchestration loop finished. Best validation loss: {best_loss:.4f}")

# Start training
run_finetuning(config)